In [11]:
import re
from typing import Optional
import os

from google.api_core.client_options import ClientOptions
from google.api_core.exceptions import InternalServerError
from google.api_core.exceptions import RetryError
from google.cloud import documentai  # type: ignore
from google.cloud import storage

import json

In [2]:

project_id = ""
location = "us" # Format is "us" or "eu"
processor_id = "" # Create processor before running sample
gcs_output_uri = "gs://react_bucket_wf/react_data/output/" # Must end with a trailing slash `/`. Format: gs://bucket/directory/subdirectory/
processor_version_id = "pretrained-layout-parser-v1.0-2024-06-03" # Optional. Example: pretrained-ocr-v1.0-2020-09-23

gcs_input_uri = "gs://react_bucket_wf/react_data/public_auditor_notes/naf23.pdf" # Format: gs://bucket/directory/file.pdf
input_mime_type = "application/pdf"
gcs_input_prefix = "gs://react_bucket_wf/react_data/public_auditor_notes" # Format: gs://bucket/directory/



In [32]:
def batch_process_documents(
    project_id: str,
    location: str,
    processor_id: str,
    gcs_output_uri: str,
    process_options,
    processor_version_id: Optional[str] = None,
    gcs_input_uri: Optional[str] = None,
    input_mime_type: Optional[str] = None,
    gcs_input_prefix: Optional[str] = None,
    field_mask: Optional[str] = None,
    timeout: int = 400,
    
) -> None:
    # You must set the `api_endpoint` if you use a location other than "us".
    opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")

    
    
    client = documentai.DocumentProcessorServiceClient(client_options=opts)

    if gcs_input_uri:
        # Specify specific GCS URIs to process individual documents
        gcs_document = documentai.GcsDocument(
            gcs_uri=gcs_input_uri, mime_type=input_mime_type
        )
        # Load GCS Input URI into a List of document files
        gcs_documents = documentai.GcsDocuments(documents=[gcs_document])
        input_config = documentai.BatchDocumentsInputConfig(gcs_documents=gcs_documents)
    else:
        # Specify a GCS URI Prefix to process an entire directory
        gcs_prefix = documentai.GcsPrefix(gcs_uri_prefix=gcs_input_prefix)
        input_config = documentai.BatchDocumentsInputConfig(gcs_prefix=gcs_prefix)

    # Cloud Storage URI for the Output Directory
    gcs_output_config = documentai.DocumentOutputConfig.GcsOutputConfig(
        gcs_uri=gcs_output_uri, field_mask=field_mask
    )

    # Where to write results
    output_config = documentai.DocumentOutputConfig(gcs_output_config=gcs_output_config)

    if processor_version_id:
        # The full resource name of the processor version, e.g.:
        # projects/{project_id}/locations/{location}/processors/{processor_id}/processorVersions/{processor_version_id}
        name = client.processor_version_path(
            project_id, location, processor_id, processor_version_id
        )
    else:
        # The full resource name of the processor, e.g.:
        # projects/{project_id}/locations/{location}/processors/{processor_id}
        name = client.processor_path(project_id, location, processor_id)

    request = documentai.BatchProcessRequest(
        name=name,
        input_documents=input_config,
        document_output_config=output_config,
        process_options=process_options
    )

    # BatchProcess returns a Long Running Operation (LRO)
    operation = client.batch_process_documents(request)

    
    
    # Continually polls the operation until it is complete.
    # This could take some time for larger files
    # Format: projects/{project_id}/locations/{location}/operations/{operation_id}
    try:
        print(f"Waiting for operation {operation.operation.name} to complete...")
        operation.result(timeout=timeout)
    # Catch exception when operation doesn't finish before timeout
    except (RetryError, InternalServerError) as e:
        print(e.message)

    # NOTE: Can also use callbacks for asynchronous processing
    #
    # def my_callback(future):
    #   result = future.result()
    #
    # operation.add_done_callback(my_callback)

    # After the operation is complete,
    # get output document information from operation metadata
    metadata = documentai.BatchProcessMetadata(operation.metadata)

    if metadata.state != documentai.BatchProcessMetadata.State.SUCCEEDED:
        raise ValueError(f"Batch Process Failed: {metadata.state_message}")

    storage_client = storage.Client()

    print("Output files:")
    # One process per Input Document
    for process in list(metadata.individual_process_statuses):
        # output_gcs_destination format: gs://BUCKET/PREFIX/OPERATION_NUMBER/INPUT_FILE_NUMBER/
        # The Cloud Storage API requires the bucket name and URI prefix separately
        matches = re.match(r"gs://(.*?)/(.*)", process.output_gcs_destination)
        if not matches:
            print(
                "Could not parse output GCS destination:",
                process.output_gcs_destination,
            )
            continue

        output_bucket, output_prefix = matches.groups()

        # Get List of Document Objects from the Output Bucket
        output_blobs = storage_client.list_blobs(output_bucket, prefix=output_prefix)

        # Document AI may output multiple JSON files per source file
        for blob in output_blobs:
            # Document AI should only output JSON files to GCS
            if blob.content_type != "application/json":
                print(
                    f"Skipping non-supported file: {blob.name} - Mimetype: {blob.content_type}"
                )
                continue

            # Download JSON File as bytes object and convert to Document Object
            print(f"Fetching {blob.name}")
            document = documentai.Document.from_json(
                blob.download_as_bytes(), ignore_unknown_fields=True
            )

            # For a full list of Document object attributes, please reference this page:
            # https://cloud.google.com/python/docs/reference/documentai/latest/google.cloud.documentai_v1.types.Document

            # Read the text recognition output from the processor
            # print("The document contains the following text:")
            # print(document.text)
            
    return operation
        

In [33]:

process_options = documentai.ProcessOptions(
        layout_config=documentai.ProcessOptions.LayoutConfig(
            chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                chunk_size=2000,
                include_ancestor_headings=True,
            )
        )
    )

operation=batch_process_documents(
    project_id=project_id,
    location=location,
    processor_id=processor_id,
    gcs_output_uri=gcs_output_uri,
    processor_version_id=processor_version_id,
    gcs_input_uri=gcs_input_uri,
    input_mime_type=input_mime_type,
    gcs_input_prefix=gcs_input_prefix,
    process_options=process_options
)

Waiting for operation projects/984528513541/locations/us/operations/3038173120623511348 to complete...
Output files:
Fetching react_data/output/3038173120623511348/0/naf23-0.json


In [34]:
operation

In [35]:
metadata = documentai.BatchProcessMetadata(operation.metadata)
print(metadata.state)

State.SUCCEEDED


In [ ]:
storage_client = storage.Client()

print("Output files:")
# One process per Input Document

text=None

for process in list(metadata.individual_process_statuses):
    print("-"*20, 'process details', "-"*20)
    print(process)
    print("-"*20, 'closing process details', "-"*20)
    print()
    
    # output_gcs_destination format: gs://BUCKET/PREFIX/OPERATION_NUMBER/INPUT_FILE_NUMBER/
    # The Cloud Storage API requires the bucket name and URI prefix separately
    
    matches = re.match(r"gs://(.*?)/(.*)", process.output_gcs_destination)
    print('matches',matches)
    if not matches:
        print(
            "Could not parse output GCS destination:",
            process.output_gcs_destination,
        )
        continue

    output_bucket, output_prefix = matches.groups()
    print(output_bucket,output_prefix)
    

    # Get List of Document Objects from the Output Bucket
    output_blobs = storage_client.list_blobs(output_bucket, prefix=output_prefix)
    print('blob type', type(output_blobs))
    
    # Document AI may output multiple JSON files per source file
    for blob in output_blobs:
        # Document AI should only output JSON files to GCS
        if blob.content_type != "application/json":
            print(
                f"Skipping non-supported file: {blob.name} - Mimetype: {blob.content_type}"
            )
            continue

        # Download JSON File as bytes object and convert to Document Object
        print(f"Fetching {blob.name}")
        document = documentai.Document.from_json(
            blob.download_as_bytes(), ignore_unknown_fields=True
        )

        text=json.loads(documentai.Document.to_json(document))
 
        


In [51]:
text.keys()

dict_keys(['shardInfo', 'documentLayout', 'chunkedDocument', 'mimeType', 'text', 'textStyles', 'pages', 'entities', 'entityRelations', 'textChanges', 'revisions'])

In [65]:
text['chunkedDocument']['chunks'][0]['content']

"# Annual Financial Report\n\nNational Ataxia Foundation St. Louis Park, Minnesota For the years ended December 31, 2023 and 2022 Abdo Lighting the path forward Edina Office 5201 Eden Avenue, Ste 250 Edina, MN 55436 P 952.835.9090 Mankato Office 100 Warren Street, Ste 600 Mankato, MN 56001 P 507.625.2727 Scottsdale Office 14500 N Northsight Blvd, Ste 233 Scottsdale, AZ 85260 P 480.864.5579 National Ataxia Foundation Table of Contents December 31, 2023 and 2022 2\n\n|-|-|\n|  | Page No. |\n| Independent Auditor's Report | 3 |\n| Financial Statements |  |\n| Statements of Financial Position | 6 |\n| Statements of Activities | 7 |\n| Statements of Functional Expenses | 9 |\n| Statements of Cash Flows | 11 |\n| Notes to the Financial Statements | 12 |\n\nAbdo\n\n## INDEPENDENT AUDITOR'S REPORT\n\nBoard of Directors National Ataxia Foundation St. Louis Park, Minnesota\n\n## Opinion\n\nWe have audited the accompanying financial statements of National Ataxia Foundation (the Foundation), which

### load directly from bucket of the processed json

In [82]:
output_bucket_name='react_bucket_wf' 
output_prefix='react_data/output/3038173120623511348/0'

In [89]:
# Get List of Document Objects from the Output Bucket
storage_client = storage.Client()
output_blobs = storage_client.list_blobs(output_bucket, prefix=output_prefix)
print('blob type', type(output_blobs))

# Document AI may output multiple JSON files per source file
for blob in output_blobs:
    # Document AI should only output JSON files to GCS
    if blob.content_type != "application/json":
        print(
            f"Skipping non-supported file: {blob.name} - Mimetype: {blob.content_type}"
        )
        continue
        
    if blob.name.endswith(".jsonl"):
        continue
    print(f"Fetching {blob.name}")
    
    # document = documentai.Document.from_json(
    #     blob.download_as_bytes(), ignore_unknown_fields=True
    # )
    # pdf_json = json.loads(documentai.Document.to_json(document))
    
    document=blob.download_as_bytes()
    pdf_json = json.loads(document)
    


blob type <class 'google.api_core.page_iterator.HTTPIterator'>
Fetching react_data/output/3038173120623511348/0/naf23-0.json


In [90]:
pdf_json.keys()

dict_keys(['chunkedDocument', 'documentLayout', 'shardInfo'])

In [91]:
pdf_json['chunkedDocument'].keys()

dict_keys(['chunks'])

In [92]:
# 'chunkedDocument' contains a list of chunks
len(pdf_json['chunkedDocument']['chunks'])

5

In [93]:
# each 'chunk' has the following items, 'content' has the actual content
print(pdf_json['chunkedDocument']['chunks'][0].keys())
for k in pdf_json['chunkedDocument']['chunks'][0].keys():
    if k=='content':
        continue
    print(k, pdf_json['chunkedDocument']['chunks'][0][k])
    print()

dict_keys(['chunkId', 'content', 'pageFooters', 'pageHeaders', 'pageSpan'])
chunkId c1

pageFooters [{'pageSpan': {'pageEnd': 3, 'pageStart': 3}, 'text': 'Lighting the path forward'}, {'pageSpan': {'pageEnd': 4, 'pageStart': 4}, 'text': 'Abdo Solutions.com'}, {'pageSpan': {'pageEnd': 6, 'pageStart': 6}, 'text': "See Independent Auditor's Report and Notes to the Financial Statements. 6"}, {'pageSpan': {'pageEnd': 7, 'pageStart': 7}, 'text': "See Independent Auditor's Report and Notes to the Financial Statements. 7"}, {'pageSpan': {'pageEnd': 8, 'pageStart': 8}, 'text': "See Independent Auditor's Report and Notes to the Financial Statements. 8"}]

pageHeaders [{'pageSpan': {'pageEnd': 3, 'pageStart': 3}, 'text': 'Abdo Solutions.com'}]

pageSpan {'pageEnd': 8, 'pageStart': 1}



In [12]:
filename=blob.name
print(filename)

react_data/output/3038173120623511348/0/naf23-0.json


In [55]:
print('blob name:',blob.name)
output_json_fileName=os.path.basename(blob.name)
print('output_json_fileName:',output_json_fileName)
output_json_path=os.path.dirname(blob.name)
print('output_json_path:', output_json_path)
fileNum=output_json_path.split('/')[-2]
print('fileNum:', fileNum)
output_jsonl_path=os.path.join(output_json_path, output_json_fileName.replace(".json",".jsonl"))
print('output_jsonl_path:', output_jsonl_path)


blob name: react_data/output/3038173120623511348/0/naf23-0.json
output_json_fileName: naf23-0.json
output_json_path: react_data/output/3038173120623511348/0
fileNum: 3038173120623511348
output_jsonl_path: react_data/output/3038173120623511348/0/naf23-0.jsonl


In [56]:
jsonl_output_file= ""

for chunk in pdf_json['chunkedDocument']['chunks']:
    print('chunkID:',chunk['chunkId'])
    
    jsonl_line=json.dumps({
        "chunkID": chunk['chunkId'],
        "fileName": output_json_fileName,
        "fileNum": fileNum,
        "pageStart": chunk['pageSpan']['pageStart'],
        "pageEnd": chunk['pageSpan']['pageEnd'],
        "content": chunk['content']
    })
    
    if jsonl_line:
        jsonl_output_file += jsonl_line+"\n"
        
        
lines=jsonl_output_file.splitlines()
jsonl_output_file="\n".join(line for line in lines if line.strip())    

chunkID: c1
chunkID: c2
chunkID: c3
chunkID: c4
chunkID: c5


In [59]:
len(jsonl_output_file.split("\n"))

5

In [65]:
output_bucket = storage_client.get_bucket(output_bucket_name)
output_blob = output_bucket.blob(output_jsonl_path)
output_blob.upload_from_string(jsonl_output_file, content_type="application/json")

# read from jsonl on gcp to text chunks

In [67]:
bucket_name='react_bucket_wf' 
file_prefix='react_data/output/3038173120623511348/0'

In [80]:
storage_client = storage.Client()
blobs = storage_client.list_blobs(bucket_name, prefix=file_prefix)
for blob in blobs:
    if blob.content_type != "application/json":
        continue
    
    
    if blob.name.endswith(".json"):
        continue
    print(f"current file name: {blob.name}")    
    
    
    blob_bytes = blob.download_as_bytes()
    jsonl_content = blob_bytes.decode('utf-8')
    json_lines = jsonl_content.strip().split('\n')

    
    text_chunks = []
    for line in json_lines:
        if line:  # Ensure the line is not empty
            json_object = json.loads(line)
            text_chunks.append(json_object['content'])

    for text_chunk in text_chunks:
        print(text_chunk)
        print()
    
    
    


current file name: react_data/output/3038173120623511348/0/naf23-0.json
current file name: react_data/output/3038173120623511348/0/naf23-0.jsonl
# Annual Financial Report

National Ataxia Foundation St. Louis Park, Minnesota For the years ended December 31, 2023 and 2022 Abdo Lighting the path forward Edina Office 5201 Eden Avenue, Ste 250 Edina, MN 55436 P 952.835.9090 Mankato Office 100 Warren Street, Ste 600 Mankato, MN 56001 P 507.625.2727 Scottsdale Office 14500 N Northsight Blvd, Ste 233 Scottsdale, AZ 85260 P 480.864.5579 National Ataxia Foundation Table of Contents December 31, 2023 and 2022 2

|-|-|
|  | Page No. |
| Independent Auditor's Report | 3 |
| Financial Statements |  |
| Statements of Financial Position | 6 |
| Statements of Activities | 7 |
| Statements of Functional Expenses | 9 |
| Statements of Cash Flows | 11 |
| Notes to the Financial Statements | 12 |

Abdo

## INDEPENDENT AUDITOR'S REPORT

Board of Directors National Ataxia Foundation St. Louis Park, Minnesot

In [ ]:
import json

# Path to your JSONL file
jsonl_file_path = 'path/to/your/file.jsonl'

# List to store the parsed JSON objects
data_list = []

# Open the JSONL file and read line by line
with open(jsonl_file_path, 'r') as file:
    for line in file:
        # Parse the JSON from the line
        json_object = json.loads(line.strip())
        data_list.append(json_object)  # Add to the list or process as needed

# Optionally, print the parsed data
for item in data_list:
    print(item)
